# Calibrate all snapshots (BTC and ETH) → Excel workbook

This notebook runs the **batch calibration pipeline** that calibrates **Black**, **Heston**, and **SVCJ** to all
`deribit_options_snapshot_*.csv` files in `data/` and persists results to a single Excel workbook.

Outputs in `calibration_results.xlsx`:

- `black_params`, `heston_params`, `svcj_params`: one row per (snapshot timestamp, currency)
- `train_data`, `test_data`: the option rows used, each with model price columns

Key features:

- **Resume-safe**: if the workbook exists, each currency resumes from the first snapshot after the latest timestamp
  already present in `black_params`.
- **Crash-safe writes**: workbook flush is atomic (temp file → replace).
- **Warm-start**: each worker chunk warm-starts from the most recent successful params prior to the chunk start.
- **Threading**: worker threads may finish out-of-order, but commits are written in chronological order.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

NOTEBOOK_ROOT = Path.cwd().resolve()
if (NOTEBOOK_ROOT / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT
elif (NOTEBOOK_ROOT / "notebooks" / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT / "notebooks"
else:
    raise FileNotFoundError(f"Could not locate notebooks/_lib from {NOTEBOOK_ROOT}")

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _lib.common import configure_notebook_display, locate_notebook_paths

PATHS = locate_notebook_paths(NOTEBOOK_DIR)
ROOT = PATHS.project_root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.batch_runner import BatchConfig, run_all_snapshots_to_excel
from src.calibration import WeightConfig
from src.inverse_fft_pricer import FFTParams

configure_notebook_display(max_columns=200, width=180)
print("Project root:", ROOT)


Project root: /Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-clean


In [2]:
OUTPUT_XLSX = ROOT / "excel files/calibration_results_reg_1000.xlsx"

cfg = BatchConfig(
    project_root=ROOT,
    output_xlsx=OUTPUT_XLSX,
    resume=True,
    save_every_n_files=150,
    smoke_test_max_files_per_currency=None,
    filter_rules=dict(
        require_bid_ask=True,
        min_time_to_maturity=1/365,
        max_time_to_maturity=None,
        min_open_interest=1.0,
        min_vega=0.0,
        max_rel_spread=0.50,
        moneyness_range=(0.5, 2.0),
        drop_synthetic_underlyings=False,
    ),
    min_options_after_filter=50,
    weight_config=WeightConfig(
        use_spread=True,
        use_vega=False,
        use_open_interest=False,
        spread_power=1.0,
        vega_power=0.5,
        oi_power=0.5,
        eps_spread=1e-6,
        eps_other=1e-12,
        cap=1e6,
    ),
    fft_base=FFTParams(
        N=2**12,
        eta=0.10,
        alpha=1.5,
        b=-10.0,
        use_simpson=True,
    ),
    train_frac=0.70,
    global_random_seed=123,
    max_nfev=dict(black=250, heston=250, svcj=250),
    l2_prev_strength=1000.0,
    runtime_top_expiries_by_oi=None,
    runtime_max_options=None,
    n_workers=int(os.cpu_count() - 2),
    limit_internal_threads=True,
    internal_num_threads=1,
    currencies=("BTC", "ETH"),
    verbose=True,
)


In [3]:
wb = run_all_snapshots_to_excel(cfg)
print("Done. Workbook at:", cfg.output_xlsx)


[BTC] Nothing to do (resume found up-to-date workbook).
[ETH] Nothing to do (resume found up-to-date workbook).
Done. Workbook at: /Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-clean/excel files/calibration_results_reg_1000.xlsx
